In [1]:
# ── Cell 1 ──
!pip install pyvis pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 12.0 MB/s eta 0:00:00


In [ ]:
# ── Cell 2 ──
import pandas as pd, re, json
from pyvis.network import Network
from google.colab import files

# Upload CSV (or set CSV_PATH to a Drive/GitHub raw path instead)
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]

COLOR_MAP = {
    "PI": "#e63946", "EA Element": "#f1a208", "Model": "#2a9d8f",
    "Framework": "#264653", "Inference": "#e76f51", "Output": "#8ecae6",
    "Temporal": "#ffb703", "Temporal_2": "#fb8500", "Prediction scale": "#6a4c93",
    "End users": "#4361ee", "End user sector": "#4cc9f0", "Synergies": "#7209b7",
}
ATTR_COLS = [c for c in COLOR_MAP if c != "PI"]

def clean_val(v):
    """Split multi-entry cells on commas (ignoring commas inside parens), strip cruft."""
    if pd.isna(v):
        return []
    parts = re.split(r",(?![^()]*\))", str(v))
    out = []
    for p in parts:
        p = p.strip(" ,")
        p = re.sub(r"^\)|\)$", "", p).strip()
        p = re.sub(r"^and\s+", "", p, flags=re.I).strip()
        if p:
            out.append(p)
    return out

df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

net = Network(height="900px", width="100%", bgcolor="#111111", font_color="white",
              directed=False, notebook=False, cdn_resources="in_line")
net.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=120,
               spring_strength=0.02, damping=0.09)

added = {}
def add_node(label, group):
    key = f"{group}::{label}"
    if key not in added:
        net.add_node(key, label=label, group=group, title=f"{group}: {label}",
                      color=COLOR_MAP.get(group, "#999999"))
        added[key] = True
    return key

for _, row in df.iterrows():
    pi_vals = clean_val(row.get("PI"))
    if not pi_vals:
        continue
    pi_key = add_node(pi_vals[0], "PI")
    for col in ATTR_COLS:
        for val in clean_val(row.get(col)):
            node_key = add_node(val, col)
            net.add_edge(pi_key, node_key, title=col)

net.set_options(json.dumps({
    "nodes": {"shape": "dot", "size": 14, "font": {"size": 14, "color": "#ffffff"}},
    "edges": {"color": {"color": "#555555", "opacity": 0.5}, "smooth": False},
    "physics": {"stabilization": {"iterations": 200}},
    "interaction": {"hover": True, "tooltipDelay": 100, "navigationButtons": True, "keyboard": True}
}))

OUT_HTML = "network_viz.html"
net.write_html(OUT_HTML, notebook=False)

# ── inject filter/search/physics control panel ──
groups = list(COLOR_MAP.keys())
checkboxes = "".join(
    f'<label style="display:block;margin:2px 0;">'
    f'<input type="checkbox" checked data-group="{g}" '
    f'onchange="toggleGroup(\'{g}\', this.checked)"> '
    f'<span style="color:{COLOR_MAP[g]}">●</span> {g}</label>'
    for g in groups
)

panel = f"""
<div id="controls" style="position:fixed;top:10px;left:10px;z-index:999;
background:rgba(20,20,20,0.92);color:#fff;padding:14px;border-radius:8px;
font-family:sans-serif;font-size:13px;max-height:90vh;overflow-y:auto;width:230px;">
  <b>Filter Groups</b><hr>
  {checkboxes}
  <hr>
  <input id="searchBox" placeholder="Search node..."
    style="width:100%;margin-bottom:6px;box-sizing:border-box;">
  <button onclick="doSearch()">Find</button>
  <button onclick="resetView()">Reset</button><br><br>
  <label><input type="checkbox" id="physicsToggle" checked onchange="togglePhysics()"> Physics</label>
</div>
<script>
function toggleGroup(g, checked) {{
  var ids = nodes.getIds().filter(id => nodes.get(id).group === g);
  nodes.update(ids.map(id => ({{id: id, hidden: !checked}})));
}}
function togglePhysics() {{
  network.setOptions({{physics: {{enabled: document.getElementById('physicsToggle').checked}}}});
}}
function doSearch() {{
  var q = document.getElementById('searchBox').value.toLowerCase();
  var match = nodes.get().find(n => n.label.toLowerCase().includes(q));
  if (match) {{
    network.selectNodes([match.id]);
    network.focus(match.id, {{scale: 1.5, animation: true}});
  }}
}}
function resetView() {{
  nodes.update(nodes.getIds().map(id => ({{id: id, hidden: false}})));
  document.querySelectorAll('#controls input[data-group]').forEach(cb => cb.checked = true);
  document.getElementById('physicsToggle').checked = true;
  network.setOptions({{physics: {{enabled: true}}}});
  network.fit();
}}
</script>
"""

with open(OUT_HTML, "r", encoding="utf-8") as f:
    html = f.read()
html = html.replace("</body>", panel + "</body>")
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

files.download(OUT_HTML)
print(f"Wrote {OUT_HTML}, nodes={len(added)}, edges={len(net.edges)}")